## Resources on CIE chromaticity:
- https://www.pbr-book.org/4ed/Radiometry,_Spectra,_and_Color/Color
- https://www.itu.int/dms_pubrec/itu-r/rec/bt/R-REC-BT.2100-3-202502-I!!PDF-C.pdf
- https://registry.khronos.org/DataFormat/specs/1.3/dataformat.1.3.html#PRIMARY_CONVERSION

In [7]:
import numpy as np
from IPython.display import Math, display

np.set_printoptions(precision=9, suppress=True)


def xy_to_xyz_direction(xy):
    """Return the unscaled XYZ direction for a CIE xy chromaticity."""
    x, y = xy
    return np.array([x / y, 1.0, (1.0 - x - y) / y], dtype=np.float64)


def primaries_to_matrices(red, green, blue, white):
    """Build P, S, RGB->XYZ M, and XYZ->RGB M_inv from xy primaries."""
    r = xy_to_xyz_direction(red)
    g = xy_to_xyz_direction(green)
    b = xy_to_xyz_direction(blue)
    w = xy_to_xyz_direction(white)

    # Columns are the unscaled RGB primary directions in XYZ space.
    P = np.column_stack([r, g, b])

    # Scale the three primary directions so RGB=(1,1,1) lands on white.
    S = np.linalg.solve(P, w) # PS = w
    M = P @ np.diag(S)
    M_inv = np.linalg.inv(M)
    return P, S, M, M_inv


def latex_matrix(value, precision=6):
    """Format a numpy vector or matrix as a LaTeX bmatrix."""
    a = np.asarray(value)
    if a.ndim == 1:
        a = a.reshape(-1, 1)
    rows = []
    for row in a:
        rows.append(" & ".join(f"{x:.{precision}f}" for x in row))
    return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"


def show_matrix(symbol, value, precision=6):
    display(Math(fr"{symbol} = {latex_matrix(value, precision)}"))


def calc(name, primaries):
    P, S, M, M_inv = primaries_to_matrices(**primaries)
    label = name.replace(" ", r"\,")
    display(Math(fr"\text{{{name}}}"))
    show_matrix(fr"P_{{{label}}}", P)
    show_matrix(fr"S_{{{label}}}", S)
    show_matrix(fr"M_{{{label}\to XYZ}}", M)
    show_matrix(fr"M^{{-1}}_{{XYZ\to {label}}}", M_inv)
    return P, S, M, M_inv


BT709 = {
    "red":   (0.640, 0.330),
    "green": (0.300, 0.600),
    "blue":  (0.150, 0.060),
    "white": (0.3127, 0.3290),  # D65
}

BT2020 = {
    "red":   (0.708, 0.292),
    "green": (0.170, 0.797),
    "blue":  (0.131, 0.046),
    "white": (0.3127, 0.3290),  # D65
}

_, _, M709, M709_inv = calc("BT.709 / sRGB", BT709)
_, _, M2020, M2020_inv = calc("BT.2020", BT2020)

# Optional sanity checks.
white = np.ones(3)
print()
print("709 white -> XYZ:", M709 @ white)
print("2020 white -> XYZ:", M2020 @ white)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


709 white -> XYZ: [0.950455927 1.          1.089057751]
2020 white -> XYZ: [0.950455927 1.          1.089057751]


## White Point Conversion

Reference: [Bruce Lindbloom, Chromatic Adaptation](http://www.brucelindbloom.com/index.html?Eqn_ChromAdapt.html)

The RGB-to-XYZ matrix for a set of primaries is built relative to that color space's reference white. If two RGB spaces use the same white point, a primary conversion is just:

$$
M_{A \to B} = M^{-1}_{B \to XYZ} M_{A \to XYZ}
$$

When the white points differ, insert a chromatic adaptation transform in XYZ space:

$$
M_{A \to B} = M^{-1}_{B \to XYZ} C_{W_A \to W_B} M_{A \to XYZ}
$$

Chromatic adaptation first maps XYZ into a cone-response domain using a method matrix $M_A$, scales the source white to the destination white in that domain, then maps back:

$$
C_{W_A \to W_B} = M_A^{-1}\,\mathrm{diag}\left(\frac{M_A W_B}{M_A W_A}\right)M_A
$$

where $W_A$ and $W_B$ are XYZ white points normalized to $Y=1$. Lindbloom lists three common choices for $M_A$: XYZ Scaling, Bradford, and Von Kries.



In [8]:
XYZ_SCALING = np.eye(3, dtype=np.float64)
XYZ_SCALING_INV = np.eye(3, dtype=np.float64)

BRADFORD = np.array([
    [ 0.8951000,  0.2664000, -0.1614000],
    [-0.7502000,  1.7135000,  0.0367000],
    [ 0.0389000, -0.0685000,  1.0296000],
], dtype=np.float64)
BRADFORD_INV = np.array([
    [ 0.9869929, -0.1470543,  0.1599627],
    [ 0.4323053,  0.5183603,  0.0492912],
    [-0.0085287,  0.0400428,  0.9684867],
], dtype=np.float64)

VON_KRIES = np.array([
    [ 0.4002400,  0.7076000, -0.0808100],
    [-0.2263000,  1.1653200,  0.0457000],
    [ 0.0000000,  0.0000000,  0.9182200],
], dtype=np.float64)
VON_KRIES_INV = np.array([
    [ 1.8599364, -1.1293816,  0.2198974],
    [ 0.3611914,  0.6388125, -0.0000064],
    [ 0.0000000,  0.0000000,  1.0890636],
], dtype=np.float64)

CAT_METHODS = {
    "XYZ Scaling": (XYZ_SCALING, XYZ_SCALING_INV),
    "Bradford": (BRADFORD, BRADFORD_INV),
    "Von Kries": (VON_KRIES, VON_KRIES_INV),
}


def chromatic_adaptation_matrix(src_white, dst_white, cat=BRADFORD, cat_inv=BRADFORD_INV):
    """Return an XYZ chromatic adaptation matrix from src white to dst white."""
    src_xyz = xy_to_xyz_direction(src_white)
    dst_xyz = xy_to_xyz_direction(dst_white)
    src_cones = cat @ src_xyz
    dst_cones = cat @ dst_xyz
    return cat_inv @ np.diag(dst_cones / src_cones) @ cat


ACES_D60 = (0.32168, 0.33767)
AP1 = {
    "red":   (0.713, 0.293),
    "green": (0.165, 0.830),
    "blue":  (0.128, 0.044),
    "white": ACES_D60,
}

_, _, MAP1, MAP1_inv = calc("ACES AP1", AP1)

for name, (cat, cat_inv) in CAT_METHODS.items():
    label = name.replace(" ", r"\,")
    show_matrix(fr"M_A^{{{label}}}", cat)
    show_matrix(fr"(M_A^{{{label}}})^{{-1}}", cat_inv)

CAT_D65_to_D60_BFD = chromatic_adaptation_matrix(BT709["white"], AP1["white"])
CAT_D60_to_D65_BFD = chromatic_adaptation_matrix(AP1["white"], BT709["white"])

M709_to_AP1_BFD = MAP1_inv @ CAT_D65_to_D60_BFD @ M709
MAP1_to_709_BFD = M709_inv @ CAT_D60_to_D65_BFD @ MAP1

show_matrix(r"C_{D65\to D60}^{BFD}", CAT_D65_to_D60_BFD)
show_matrix(r"M_{BT.709/sRGB\to AP1}^{BFD}", M709_to_AP1_BFD)
show_matrix(r"M_{AP1\to BT.709/sRGB}^{BFD}", MAP1_to_709_BFD)

white = np.ones(3)
print("BT.709 white -> AP1:", M709_to_AP1_BFD @ white)
print("AP1 white -> BT.709:", MAP1_to_709_BFD @ white)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

BT.709 white -> AP1: [1.          1.000000049 0.999999948]
AP1 white -> BT.709: [0.999999991 1.000000047 0.99999994 ]
